# 📋 NOTEBOOK 10 : TABLEAUX HAUTE QUALITÉ POUR PUBLICATION

## Contexte
Génération de **tableaux publication-ready** au format LaTeX, Markdown et CSV propres,
respectant les standards des revues scientifiques.

## Tableaux générés
| Table | Titre | Contenu |
|-------|-------|---------|
| **Table 1** | Statistiques descriptives | μ, σ, min, Q25, Q50, Q75, max par feature clé |
| **Table 2** | Performances comparées | R², RMSE, MAE pour top 10 modèles |
| **Table 3** | Validation statistique | Bootstrap IC95%, tests Wilcoxon, p-values |
| **Table 4** | Feature importance | Top 15 features + scores d'importance |
| **Table 5** | Analyse par classe | Classification rendement faible/moyen/fort |

## Formats exportés
- **CSV** : import facile Excel/LibreOffice
- **LaTeX** : intégration directe manuscripts (.tex)
- **Markdown** : preview GitHub, documentation

## Références stylistiques
- APA 7th Edition (American Psychological Association)
- LaTeX : booktabs package (toprule, midrule, bottomrule)


In [1]:
# ============================================================
# 📦 IMPORTS ET CONFIGURATION
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

for d in ['../results/tables_publication']:
    os.makedirs(d, exist_ok=True)

print("=" * 80)
print("✅ NOTEBOOK 10 : TABLEAUX PUBLICATION — Configuration OK")
print("=" * 80)
print(f"   Dossier sortie : ../results/tables_publication/")
print("=" * 80)

✅ NOTEBOOK 10 : TABLEAUX PUBLICATION — Configuration OK
   Dossier sortie : ../results/tables_publication/


In [2]:
# ============================================================
# 1. CHARGEMENT DES DONNÉES ET RÉSULTATS
# ============================================================
print("\n" + "=" * 80)
print("📥 CHARGEMENT")
print("=" * 80)

# Données
df_raw  = pd.read_csv('../data/processed/data_clean.csv')
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Résultats combinés
results_all = pd.read_csv('../results/07_all_models_combined.csv')
results_all = results_all.sort_values('R2_Test', ascending=False).reset_index(drop=True)

# Meilleur modèle — chargé dynamiquement
best_model_name = results_all.iloc[0]['Modèle'].strip()
model_files_map = {
    'Linear':                   '../models/linear_regression.pkl',
    'Ridge':                    '../models/ridge_regression.pkl',
    'Lasso':                    '../models/lasso_regression.pkl',
    'ElasticNet':               '../models/elasticnet_regression.pkl',
    'Decision Tree (opt)':      '../models/decision_tree.pkl',
    'Random Forest (opt)':      '../models/random_forest.pkl',
    'Gradient Boosting (opt)':  '../models/gradient_boosting.pkl',
    'LightGBM (opt)':           '../models/lightgbm.pkl',
    'CatBoost (opt)':           '../models/catboost.pkl',
    'HistGradientBoosting':     '../models/histgradient.pkl',
    'AdaBoost':                 '../models/adaboost.pkl',
    'KNN (Optuna)':             '../models/knn.pkl',
    'LinearSVR (Optuna)':       '../models/linearsvr.pkl',
    'SGD (Optuna)':             '../models/sgd.pkl',
}
model_path = model_files_map.get(best_model_name)
if model_path is None or not os.path.exists(model_path):
    raise FileNotFoundError(f"Modèle '{best_model_name}' introuvable : {model_path}")

best_model  = joblib.load(model_path)
y_test_pred = best_model.predict(X_test)

print(f"✅ Données brutes    : {len(df_raw)} observations")
print(f"✅ Features ML       : {X_train.shape[1]}")
print(f"✅ Train/Test        : {X_train.shape[0]} / {X_test.shape[0]}")
print(f"✅ Meilleur modèle   : {best_model_name}")
print(f"   Type             : {type(best_model).__name__}")
print(f"   Fichier          : {model_path}")
print(f"✅ {len(results_all)} modèles dans les résultats")


📥 CHARGEMENT
✅ Données brutes    : 3803 observations
✅ Features ML       : 43
✅ Train/Test        : 3042 / 761
✅ Meilleur modèle   : Gradient Boosting (opt)
   Type             : GradientBoostingRegressor
   Fichier          : ../models/gradient_boosting.pkl
✅ 14 modèles dans les résultats


In [3]:
# ============================================================
# 2. TABLE 1 : STATISTIQUES DESCRIPTIVES
# ============================================================
print("\n" + "=" * 80)
print("📊 TABLE 1 : STATISTIQUES DESCRIPTIVES")
print("=" * 80)

key_vars       = ['yield', 'area', 'rain_cycle', 'nitrogen', 'manure', 'urea', 'npk']
key_vars_avail = [v for v in key_vars if v in df_raw.columns]

if not key_vars_avail:
    print("⚠️  Aucune variable clé trouvée — utilisation des colonnes numériques")
    key_vars_avail = df_raw.select_dtypes(include=[np.number]).columns.tolist()[:7]

rows = []
for var in key_vars_avail:
    data = df_raw[var].dropna()
    rows.append({
        'Variable':     var,
        'n':            len(data),
        'Moyenne':      data.mean(),
        'Écart-type':   data.std(),
        'Min':          data.min(),
        'Q25':          data.quantile(0.25),
        'Médiane':      data.median(),
        'Q75':          data.quantile(0.75),
        'Max':          data.max(),
        'CV (%)':       data.std() / data.mean() * 100
    })
table1 = pd.DataFrame(rows).round(2)

print("\n📋 Table 1 — Statistiques descriptives :")
print(table1.to_string(index=False))

os.makedirs('../results/tables_publication', exist_ok=True)
table1.to_csv('../results/tables_publication/Table1_descriptives.csv', index=False)
table1.to_latex('../results/tables_publication/Table1_descriptives.tex', index=False,
                caption='Statistiques descriptives des variables clés',
                label='tab:descriptives', escape=False)
try:
    table1.to_markdown('../results/tables_publication/Table1_descriptives.md', index=False)
    print("\n✅ Table1_descriptives.csv / .tex / .md")
except ImportError:
    print("\n✅ Table1_descriptives.csv / .tex")
    print("   ⚠️  Markdown ignoré — installez tabulate : pip install tabulate")


📊 TABLE 1 : STATISTIQUES DESCRIPTIVES

📋 Table 1 — Statistiques descriptives :
  Variable    n   Moyenne  Écart-type      Min       Q25   Médiane       Q75        Max   CV (%)
     yield 3803 2566.5300    908.3200 318.0000 1915.6000 2546.7000 3142.9000  5350.0000  35.3900
      area 3803    8.8900     17.2600   0.0000    0.1000    0.5000   11.3500   260.0000 194.1200
rain_cycle 3803  919.3800    160.3300 443.4000  803.9000  925.4000  982.4000  1325.5000  17.4400
  nitrogen 3803   10.3800     17.3100   0.0000    0.0000    0.0000   18.4000    91.4500 166.7600
    manure 3803 1198.8300   1676.2800   0.0000    0.0000    0.0000 2500.0000 11976.0000 139.8300
      urea 3803   17.2600     30.2600   0.0000    0.0000    0.0000   33.3000   150.0000 175.3600
       npk 3803   22.2100     48.5800   0.0000    0.0000    0.0000    0.0000   300.0000 218.6900

✅ Table1_descriptives.csv / .tex / .md


In [4]:
# ============================================================
# 3. TABLE 2 : PERFORMANCES COMPARÉES (TOP 10 MODÈLES)
# ============================================================
print("\n" + "=" * 80)
print("📊 TABLE 2 : PERFORMANCES COMPARÉES")
print("=" * 80)

top10  = results_all.head(10).copy()
cols   = ['Modèle', 'R2_Test', 'RMSE_Test']
rename = {'Modèle': 'Modèle', 'R2_Test': 'R² (Test)', 'RMSE_Test': 'RMSE (kg/ha)'}

# MAE optionnelle
if 'MAE_Test' in top10.columns:
    cols.append('MAE_Test')
    rename['MAE_Test'] = 'MAE (kg/ha)'

table2 = top10[cols].copy().rename(columns=rename)

# Overfitting si R2_Train disponible
if 'R2_Train' in top10.columns:
    table2['Overfitting (ΔR²)'] = (top10['R2_Train'] - top10['R2_Test']).round(4)

table2 = table2.round(4)

print("\n📋 Table 2 — Performances comparées (top 10) :")
print(table2.to_string(index=False))

table2.to_csv('../results/tables_publication/Table2_model_performances.csv', index=False)
table2.to_latex('../results/tables_publication/Table2_model_performances.tex', index=False,
                caption='Performances comparées des modèles de prédiction (test set)',
                label='tab:performances', escape=False)
try:
    table2.to_markdown('../results/tables_publication/Table2_model_performances.md', index=False)
    print("\n✅ Table2_model_performances.csv / .tex / .md")
except ImportError:
    print("\n✅ Table2_model_performances.csv / .tex")


📊 TABLE 2 : PERFORMANCES COMPARÉES

📋 Table 2 — Performances comparées (top 10) :
                 Modèle  R² (Test)  RMSE (kg/ha)  MAE (kg/ha)
Gradient Boosting (opt)     0.1793      822.0410     643.3376
         LightGBM (opt)     0.1776      822.9080     641.4756
   HistGradientBoosting     0.1749      824.2692     643.6838
         CatBoost (opt)     0.1731      825.1376     644.7911
    Random Forest (opt)     0.1657      828.8442     651.2884
           KNN (Optuna)     0.1529      835.1600     654.4300
               AdaBoost     0.1339      844.4772     663.1869
    Decision Tree (opt)     0.1215      850.4843     674.8108
             ElasticNet     0.1049      858.5126          NaN
                 Linear     0.1048      858.5390          NaN

✅ Table2_model_performances.csv / .tex / .md


In [5]:
# ============================================================
# 4. TABLE 3 : VALIDATION STATISTIQUE (depuis NB09)
# ============================================================
print("\n" + "=" * 80)
print("📊 TABLE 3 : VALIDATION STATISTIQUE")
print("=" * 80)

table3_data = []

# Bootstrap (NB08)
boot_path = '../results/08_bootstrap_statistics.csv'
if os.path.exists(boot_path):
    boot_stats = pd.read_csv(boot_path)
    for _, row in boot_stats.iterrows():
        table3_data.append({
            'Test':      f'Bootstrap {row["Métrique"]} (B=1000)',
            'Valeur':    round(row['Moyenne'], 4),
            'IC95 inf':  round(row['IC95_inf'], 4),
            'IC95 sup':  round(row['IC95_sup'], 4),
            'p-value':   '—'
        })
    print(f"   ✅ Bootstrap chargé : {boot_path}")
else:
    print(f"   ⚠️  Bootstrap non trouvé : {boot_path}")

# Test de permutation (NB08)
perm_path = '../results/08_permutation_test.csv'
if os.path.exists(perm_path):
    perm_res = pd.read_csv(perm_path)
    table3_data.append({
        'Test':     'Permutation (R²)',
        'Valeur':   round(perm_res['R2_observed'].values[0], 4),
        'IC95 inf': '—',
        'IC95 sup': '—',
        'p-value':  round(perm_res['p_value'].values[0], 4)
    })
    print(f"   ✅ Test permutation chargé : {perm_path}")
else:
    print(f"   ⚠️  Permutation non trouvé : {perm_path}")

# Comparaisons Wilcoxon (NB08)
comp_path = '../results/08_model_comparisons.csv'
if os.path.exists(comp_path):
    comp = pd.read_csv(comp_path)
    for _, row in comp.iterrows():
        table3_data.append({
            'Test':     f'Wilcoxon vs {row["Modèle_concurrent"]}',
            'Valeur':   '—',
            'IC95 inf': '—',
            'IC95 sup': '—',
            'p-value':  round(row['p_value'], 4)
        })
    print(f"   ✅ Comparaisons Wilcoxon chargées : {len(comp)} modèles")
else:
    print(f"   ⚠️  Comparaisons non trouvées : {comp_path}")

if table3_data:
    table3 = pd.DataFrame(table3_data)
    print("\n📋 Table 3 — Validation statistique :")
    print(table3.to_string(index=False))

    table3.to_csv('../results/tables_publication/Table3_validation_statistics.csv', index=False)
    table3.to_latex('../results/tables_publication/Table3_validation_statistics.tex', index=False,
                    caption='Validation statistique du meilleur modèle (Bootstrap, Permutation, Wilcoxon)',
                    label='tab:validation', escape=False)
    try:
        table3.to_markdown('../results/tables_publication/Table3_validation_statistics.md', index=False)
        print("\n✅ Table3_validation_statistics.csv / .tex / .md")
    except ImportError:
        print("\n✅ Table3_validation_statistics.csv / .tex")
else:
    print("\n⚠️  Table 3 ignorée — NB08 non exécuté")
    print("   → Exécutez d'abord le notebook 08_validation.ipynb")


📊 TABLE 3 : VALIDATION STATISTIQUE
   ✅ Bootstrap chargé : ../results/08_bootstrap_statistics.csv
   ✅ Test permutation chargé : ../results/08_permutation_test.csv
   ✅ Comparaisons Wilcoxon chargées : 13 modèles

📋 Table 3 — Validation statistique :
                            Test   Valeur IC95 inf IC95 sup p-value
           Bootstrap R² (B=1000)   0.1783   0.1286   0.2255       —
         Bootstrap RMSE (B=1000) 821.9773 778.7203 868.0838       —
          Bootstrap MAE (B=1000) 643.5445 606.0618 681.2983       —
                Permutation (R²)   0.1673        —        —  0.0099
  Wilcoxon vs LinearSVR (Optuna)        —        —        —  0.0000
        Wilcoxon vs SGD (Optuna)        —        —        —  0.0000
               Wilcoxon vs Ridge        —        —        —  0.0000
              Wilcoxon vs Linear        —        —        —  0.0000
          Wilcoxon vs ElasticNet        —        —        —  0.0000
               Wilcoxon vs Lasso        —        —        —  0.0000


In [6]:
# ============================================================
# 5. TABLE 4 : FEATURE IMPORTANCE (TOP 15)
# ============================================================
print("\n" + "=" * 80)
print("📊 TABLE 4 : FEATURE IMPORTANCE")
print("=" * 80)

if hasattr(best_model, 'feature_importances_'):
    fi     = pd.Series(best_model.feature_importances_, index=X_train.columns)
    fi     = fi.sort_values(ascending=False).head(15)
    fi_pct = fi / fi.sum() * 100  # importance relative en %

    table4 = pd.DataFrame({
        'Rang':               range(1, len(fi)+1),
        'Feature':            fi.index,
        'Importance':         fi.values.round(4),
        'Importance (%)':     fi_pct.values.round(2),
        'Importance cumulée': fi_pct.cumsum().values.round(2)
    })

    print("\n📋 Table 4 — Feature Importance (top 15) :")
    print(table4.to_string(index=False))

    table4.to_csv('../results/tables_publication/Table4_feature_importance.csv', index=False)
    table4.to_latex('../results/tables_publication/Table4_feature_importance.tex', index=False,
                    caption=f'Importance des variables — {best_model_name} (top 15)',
                    label='tab:importance', escape=False)
    try:
        table4.to_markdown('../results/tables_publication/Table4_feature_importance.md', index=False)
        print("\n✅ Table4_feature_importance.csv / .tex / .md")
    except ImportError:
        print("\n✅ Table4_feature_importance.csv / .tex")

    # Vérifier aussi si SHAP disponible
    shap_path = '../results/08_shap_importance.csv'
    if os.path.exists(shap_path):
        shap_df = pd.read_csv(shap_path).head(15)
        shap_df.insert(0, 'Rang', range(1, len(shap_df)+1))
        shap_df.to_csv('../results/tables_publication/Table4b_shap_importance.csv', index=False)
        print("✅ Table4b_shap_importance.csv (SHAP)")
else:
    print("\n⚠️  Table 4 ignorée — feature_importances_ non disponible pour ce modèle")


📊 TABLE 4 : FEATURE IMPORTANCE

📋 Table 4 — Feature Importance (top 15) :
 Rang                    Feature  Importance  Importance (%)  Importance cumulée
    1      interaction_rain_area      0.1393         18.9400             18.9400
    2                  rain_year      0.0797         10.8400             29.7700
    3   interaction_ca_rain_year      0.0743         10.1000             39.8700
    4        village_Antsahamamy      0.0716          9.7300             49.6100
    5                 rain_cycle      0.0511          6.9400             56.5500
    6   village_Ambohitsilaozana      0.0490          6.6600             63.2100
    7                       area      0.0447          6.0800             69.2900
    8              soil_hillside      0.0375          5.1000             74.3900
    9            sow_day_of_year      0.0337          4.5900             78.9700
   10 interaction_till_rain_year      0.0288          3.9100             82.8900
   11  interaction_rain_nitrogen  

In [7]:
# ============================================================
# 6. TABLE 5 : ANALYSE PAR CLASSE DE RENDEMENT
# ============================================================
print("\n" + "=" * 80)
print("📊 TABLE 5 : ANALYSE PAR CLASSE DE RENDEMENT")
print("=" * 80)

q33 = y_test.quantile(0.33)
q67 = y_test.quantile(0.67)

print(f"   Seuils : Faible < {q33:.0f} kg/ha | Moyen [{q33:.0f}, {q67:.0f}] | Fort > {q67:.0f} kg/ha")

def classify(val):
    if val < q33:  return 'Faible'
    elif val < q67: return 'Moyen'
    else:           return 'Fort'

classes_true = y_test.apply(classify)
classes_pred = pd.Series(y_test_pred, index=y_test.index).apply(classify)

report = classification_report(classes_true, classes_pred,
                                output_dict=True, zero_division=0)
table5 = pd.DataFrame(report).T
table5 = table5[table5.index.isin(['Faible','Moyen','Fort',
                                    'accuracy','macro avg','weighted avg'])]
table5 = table5[['precision','recall','f1-score','support']]
table5.columns = ['Précision', 'Rappel', 'F1-score', 'Support']
table5 = table5.round(3)

print("\n📋 Table 5 — Classification par classe de rendement :")
print(table5)

table5.to_csv('../results/tables_publication/Table5_classification_report.csv')
table5.to_latex('../results/tables_publication/Table5_classification_report.tex',
                caption='Classification par classe de rendement (faible/moyen/fort)',
                label='tab:classification', escape=False)
try:
    table5.to_markdown('../results/tables_publication/Table5_classification_report.md')
    print("\n✅ Table5_classification_report.csv / .tex / .md")
except ImportError:
    print("\n✅ Table5_classification_report.csv / .tex")


📊 TABLE 5 : ANALYSE PAR CLASSE DE RENDEMENT
   Seuils : Faible < 2165 kg/ha | Moyen [2165, 2957] | Fort > 2957 kg/ha

📋 Table 5 — Classification par classe de rendement :
              Précision  Rappel  F1-score  Support
Faible           0.6410  0.3270    0.4330 251.0000
Fort             0.6150  0.2870    0.3910 251.0000
Moyen            0.3860  0.7680    0.5140 259.0000
accuracy         0.4640  0.4640    0.4640   0.4640
macro avg        0.5470  0.4610    0.4460 761.0000
weighted avg     0.5460  0.4640    0.4470 761.0000

✅ Table5_classification_report.csv / .tex / .md


In [8]:
# ============================================================
# 7. RÉSUMÉ + CHECKLIST
# ============================================================
print("\n" + "=" * 80)
print("📝 RÉSUMÉ — TABLEAUX PUBLICATION")
print("=" * 80)

summary = (
    "TABLEAUX HAUTE QUALITÉ POUR PUBLICATION\n"
    + "="*70 + "\n\n"
    + "Répertoire : ../results/tables_publication/\n\n"
    + "TABLEAUX GÉNÉRÉS (CSV + LaTeX + Markdown) :\n"
    + "  1. Table1_descriptives          — Statistiques descriptives\n"
    + "  2. Table2_model_performances    — Performances top 10 modèles\n"
    + "  3. Table3_validation_statistics — Bootstrap, Permutation, Wilcoxon\n"
    + "  4. Table4_feature_importance    — Top 15 features (+ SHAP si dispo)\n"
    + "  5. Table5_classification_report — Analyse par classe rendement\n\n"
    + "SPÉCIFICATIONS TECHNIQUES :\n"
    + "  - Format CSV    : import Excel/LibreOffice direct\n"
    + "  - Format LaTeX  : intégration manuscrit .tex (booktabs)\n"
    + "  - Format Markdown : preview GitHub, README\n\n"
    + "CONFORMITÉ :\n"
    + "  ✅ APA 7th Edition : colonnes alignées, décimales cohérentes\n"
    + "  ✅ LaTeX booktabs  : toprule, midrule, bottomrule\n"
    + "  ✅ Lisibilité      : nombres arrondis (4 décimales max)\n\n"
    + "UTILISATION LaTeX :\n"
    + "  \\input{tables_publication/Table1_descriptives.tex}\n\n"
    + "="*70 + "\n"
)

print(summary)
with open('../results/tables_publication/README_tables.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print("\n✅ ../results/tables_publication/README_tables.txt")

# ── CHECKLIST ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("✅ CHECKLIST — NOTEBOOK 10")
print("=" * 80)

checklist = [
    ("Données chargées",                        all(x in globals() for x in ['df_raw','X_train','results_all'])),
    ("Meilleur modèle chargé",                  'best_model' in globals()),
    ("Table 1 : Descriptives (CSV)",            os.path.exists('../results/tables_publication/Table1_descriptives.csv')),
    ("Table 1 : Descriptives (LaTeX)",          os.path.exists('../results/tables_publication/Table1_descriptives.tex')),
    ("Table 2 : Performances (CSV)",            os.path.exists('../results/tables_publication/Table2_model_performances.csv')),
    ("Table 2 : Performances (LaTeX)",          os.path.exists('../results/tables_publication/Table2_model_performances.tex')),
    ("Table 3 : Validation (CSV)",              os.path.exists('../results/tables_publication/Table3_validation_statistics.csv')),
    ("Table 4 : Feature Importance (CSV)",      os.path.exists('../results/tables_publication/Table4_feature_importance.csv')),
    ("Table 4 : Feature Importance (LaTeX)",    os.path.exists('../results/tables_publication/Table4_feature_importance.tex')),
    ("Table 5 : Classification (CSV)",          os.path.exists('../results/tables_publication/Table5_classification_report.csv')),
    ("Table 5 : Classification (LaTeX)",        os.path.exists('../results/tables_publication/Table5_classification_report.tex')),
    ("README_tables.txt",                       os.path.exists('../results/tables_publication/README_tables.txt')),
]

completed = 0
for item, status in checklist:
    print(f"   {'✅' if status else '❌'} {item}")
    if status: completed += 1

print(f"\n{'='*80}")
print(f"📊 PROGRESSION : {completed}/{len(checklist)} ({completed*100//len(checklist)}%)")
if completed >= 9:
    print("\n🎉 NOTEBOOK 10 TERMINÉ — Tableaux publication générés !")
    print("\n🏆 PIPELINE COMPLET TERMINÉ (NB01 → NB11)")
    print(f"   Meilleur modèle : {best_model_name}")
    print(f"   R² Test         : {results_all.iloc[0]['R2_Test']:.4f}")
    print("\n📄 FICHIERS PRÊTS POUR PUBLICATION :")
    print("   • Figures  : ../figures/publication/*.png + *.pdf")
    print("   • Tableaux : ../results/tables_publication/*.csv / .tex / .md")
    print("   • Résumés  : ../results/*_summary.txt")
else:
    missing = [item for item, status in checklist if not status]
    print(f"\n⚠️  {len(missing)} élément(s) manquant(s):")
    for item in missing:
        print(f"   ❌ {item}")
print("=" * 80)


📝 RÉSUMÉ — TABLEAUX PUBLICATION
TABLEAUX HAUTE QUALITÉ POUR PUBLICATION

Répertoire : ../results/tables_publication/

TABLEAUX GÉNÉRÉS (CSV + LaTeX + Markdown) :
  1. Table1_descriptives          — Statistiques descriptives
  2. Table2_model_performances    — Performances top 10 modèles
  3. Table3_validation_statistics — Bootstrap, Permutation, Wilcoxon
  4. Table4_feature_importance    — Top 15 features (+ SHAP si dispo)
  5. Table5_classification_report — Analyse par classe rendement

SPÉCIFICATIONS TECHNIQUES :
  - Format CSV    : import Excel/LibreOffice direct
  - Format LaTeX  : intégration manuscrit .tex (booktabs)
  - Format Markdown : preview GitHub, README

CONFORMITÉ :
  ✅ APA 7th Edition : colonnes alignées, décimales cohérentes
  ✅ LaTeX booktabs  : toprule, midrule, bottomrule
  ✅ Lisibilité      : nombres arrondis (4 décimales max)

UTILISATION LaTeX :
  \input{tables_publication/Table1_descriptives.tex}



✅ ../results/tables_publication/README_tables.txt

✅ CHECKLIST